Assignment 5: Rescue Drone Mission: Modelling and Tracking raft drifting near a rocky shoreline

Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def generate_true_trajectory(N, T, x0, v0, sigma_a):

    x_true = [x0]
    v_true = [v0]
    a_true = [0.0]

    for _ in range(1, N):
        a = np.random.normal(0, sigma_a)
        x_new = x_true[-1] + v_true[-1] * T + 0.5 * a * T**2
        v_new = v_true[-1] + a * T

        x_true.append(x_new)
        v_true.append(v_new)
        a_true.append(a)

    return np.array(x_true), np.array(v_true), np.array(a_true)


def generate_measurements(x_true, sigma_eta):

    noise = np.random.normal(0, sigma_eta, size=len(x_true))
    z = x_true + noise
    return z


def build_Phi(T):
    return np.array([[1.0, T], [0.0, 1.0]], dtype=float)


def build_G(T):
    return np.array([[0.5 * T**2],[T]], dtype=float)


def build_Q(T, sigma_a):
    G = build_G(T)
    return sigma_a**2 * (G @ G.T)


def build_H():
    return np.array([[1.0, 0.0]], dtype=float)


def build_R(sigma_eta):
    return np.array([[sigma_eta**2]], dtype=float)


def build_model_matrices(T, sigma_a, sigma_eta):
    Phi = build_Phi(T)
    G   = build_G(T)
    Q   = build_Q(T, sigma_a)
    H   = build_H()
    R   = build_R(sigma_eta)
    return Phi, G, H, Q, R


def run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0):
    Phi, G, H, Q, R = build_model_matrices(T, sigma_a, sigma_eta)
    I = np.eye(2)
    n = len(z)
    
		# Storage
    x_pred = np.zeros((n, 2))			# X_{i|i-1}
    P_pred = np.zeros((n, 2, 2))	# P_{i|i-1}
    x_filt = np.zeros((n, 2))			# X_{i|i}
    P_filt = np.zeros((n, 2, 2))	# P_{i|i}
    K_hist = np.zeros((n, 2, 1))	# K_i
    residual  = np.zeros((n, 1))	# residual_i
    S_hist = np.zeros((n, 1, 1))	# S_i

    # Init
    x_prev = np.array([x0_hat, v0_hat], dtype=float)
    P_prev = P0

    # Loop
    for i in range(n):
        # Predict
        x_pr = Phi @ x_prev								# X_{i|i-1}
        P_pr = Phi @ P_prev @ Phi.T + Q		# P_{i|i-1}

        # Filtration
        resid = z[i] - (H @ x_pr).item()	# resid_i
        S = (H @ P_pr @ H.T + R).item()		# S_i
        K = (P_pr @ H.T) / S							# K_i

        x_up = x_pr + resid * K.flatten()	# X_{i|i}
        P_up = (I - K @ H) @ P_pr					# P_{i|i}

        # Save
        x_pred[i], P_pred[i] = x_pr, P_pr
        x_filt[i], P_filt[i] = x_up, P_up
        K_hist[i], residual[i], S_hist[i] = K, resid, S

        x_prev, P_prev = x_up, P_up

    # Convenient standard deviations for analysis
    sigma_x_pred = np.sqrt(P_pred[:, 0, 0])
    sigma_x_filt = np.sqrt(P_filt[:, 0, 0])

    return {
        "Phi": Phi, "G": G, "H": H, "Q": Q, "R": R,
        "x_pred": x_pred, "P_pred": P_pred,
        "x_filt": x_filt, "P_filt": P_filt,
        "K": K_hist, "residual": residual, "S": S_hist,
        "sigma_x_pred": sigma_x_pred, "sigma_x_filt": sigma_x_filt
    }


def predict_ahead_from_filtered(x_filt, Phi, m):
    n = len(x_filt)
    Xa = np.full((n, 2), np.nan)
    Phi_m = np.linalg.matrix_power(Phi, m-1)
    for i in range(0, n-(m-1)):
        Xa[i+m-1] = (Phi_m @ x_filt[i])
    return Xa


def compute_final_error(x_true_runs, x_est_runs):
    x_true_runs = np.array(x_true_runs)
    x_est_runs = np.array(x_est_runs)
    mse = np.mean((x_true_runs - x_est_runs) ** 2, axis=0)
    rmse = np.sqrt(mse)
    
    return rmse[3:]


np.random.seed(None)

In [ ]:
fontsize_title = 30
fontsize_label = 20
fontsize_leg = 20

In [ ]:
# Parameters
N = 200
T = 1.0
sigma_a = 0.2
sigma_eta = 20
P0 = np.diag([10000.0, 10000.0])

# Initial conditions
x0 = 5.0
v0 = 1.0

# Generate true trajectory, velocity and acceleration
x_true_1, v_true_1, a_true_1 = generate_true_trajectory(N, T, x0, v0, sigma_a)
x_true_2, v_true_2, a_true_2 = generate_true_trajectory(N, T, x0, v0, sigma_a)
x_true_3, v_true_3, a_true_3 = generate_true_trajectory(N, T, x0, v0, sigma_a)

# Generate measurements
z_1 = generate_measurements(x_true_1, sigma_eta)
z_2 = generate_measurements(x_true_2, sigma_eta)
z_3 = generate_measurements(x_true_3, sigma_eta)

# Run kalman filter
out_1 = run_kalman(z_1, T, sigma_a, sigma_eta, x0, v0, P0)
out_2 = run_kalman(z_2, T, sigma_a, sigma_eta, x0, v0, P0)
out_3 = run_kalman(z_3, T, sigma_a, sigma_eta, x0, v0, P0)

In [ ]:
# Compare First true trajectory, measurements and Kalman estimate
plt.figure(figsize=(25, 8))

plt.plot(x_true_1, color='black', linewidth=1, alpha=1, label='True position x_i')
plt.plot(z_1, color='orange', linewidth=1, alpha=1, label='Measurement z_i')
plt.plot(out_1["x_filt"][:, 0], color='green', linewidth=1, alpha=1, label='Filtered x_i')

plt.title('\n Fig.1. First True trajectory, measurements and Kalman estimate \n', fontsize=fontsize_title)
plt.xlabel('\n Time step i \n', fontsize=fontsize_label)
plt.ylabel('\n Position (m) \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)
plt.show()

# Compare Second true trajectory, measurements and Kalman estimate
plt.figure(figsize=(25, 8))

plt.plot(x_true_2, color='black', linewidth=1, alpha=1, label='True position x_i')
plt.plot(z_2, color='orange', linewidth=1, alpha=1, label='Measurement z_i')
plt.plot(out_2["x_filt"][:, 0], color='green', linewidth=1, alpha=1, label='Filtered x_i')

plt.title('\n Fig.2. Second True trajectory, measurements and Kalman estimate \n', fontsize=fontsize_title)
plt.xlabel('\n Time step i \n', fontsize=fontsize_label)
plt.ylabel('\n Position (m) \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)
plt.show()

# Compare Third true trajectory, measurements and Kalman estimate
plt.figure(figsize=(25, 8))

plt.plot(x_true_3, color='black', linewidth=1, alpha=1, label='True position x_i')
plt.plot(z_3, color='orange', linewidth=1, alpha=1, label='Measurement z_i')
plt.plot(out_3["x_filt"][:, 0], color='green', linewidth=1, alpha=1, label='Filtered x_i')

plt.title('\n Fig.3. Third True trajectory, measurements and Kalman estimate \n', fontsize=fontsize_title)
plt.xlabel('\n Time step i \n', fontsize=fontsize_label)
plt.ylabel('\n Position (m) \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)
plt.show()

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(out_1["sigma_x_filt"], color='blue',  linewidth=1.5, alpha=0.8, label='Filtered std \\sigma_{x,\,i|i}')

plt.title('\n Fig.4. Standard deviation of position \n', fontsize=fontsize_title)
plt.xlabel('\n Time step i \n', fontsize=fontsize_label)
plt.ylabel('\n Standard deviation, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.show()

K = out_1["K"].squeeze(-1)

plt.figure(figsize=(25, 8))

plt.plot(K[:, 0], color='red',   linewidth=1.5, alpha=1, label='Gain for position state K_x')

plt.title('\n Fig.5. Kalman filter gain \n', fontsize=fontsize_title)
plt.xlabel('\n Time step i \n', fontsize=fontsize_label)
plt.ylabel('\n Kalman gain - position \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.show()



In [ ]:
m = 7
Xa = predict_ahead_from_filtered(out_1["x_filt"], out_1["Phi"], m)
x_ahead = Xa[:, 0]

In [ ]:
plt.figure(figsize=(25, 12))

plt.plot(x_true_1, color='black', linewidth=1.5, alpha=1, label='True trajectory x_i')
plt.plot(z_1, color='orange', linewidth=0.8, alpha=1, label='Measurements z_i')
plt.plot(x_ahead, color='blue', linewidth=1.5, alpha=1, label=fr'KF future trajectory')

plt.title('\n Fig.6. Raft position: true, measurements, and (m−1)-ahead KF prediction \n', fontsize=fontsize_title)
plt.xlabel('\n Time step i \n', fontsize=fontsize_label)
plt.ylabel('\n Raft position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)
plt.show()

In [ ]:
x0_hat = 0.0
v0_hat = 0.0
runs = 500

x_true_runs = []
x_filt_runs = []
x_pred_runs = []
x_future_runs = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    
    res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0)
    
    x_future = predict_ahead_from_filtered(res["x_filt"], res["Phi"], m=7)
    
    x_true_runs.append(x_true)
    x_filt_runs.append(res["x_filt"][:, 0])
    x_pred_runs.append(res["x_pred"][:, 0])
    x_future_runs.append(x_future[:, 0])


final_error_pred   = compute_final_error(x_true_runs, x_pred_runs)
final_error_filt   = compute_final_error(x_true_runs, x_filt_runs)
final_error_future = compute_final_error(x_true_runs, x_future_runs)

time = np.arange(len(final_error_pred))


In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_pred,   label="Predicted estimate", color="blue", linewidth=1.5, alpha=1)
plt.plot(time, final_error_filt,   label="Filtered estimate",  color="red",  linewidth=1.5, alpha=1)
plt.plot(time, final_error_future, label="Future estimate",    color="green", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values \n", fontsize=fontsize_label)
plt.title("\n Fig.7. Final errors of KF estimates \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
sigma_vector = np.full_like(final_error_filt, 20.0)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_filt, label="Filtered positions (Final error KF)", color="blue", linewidth=1.5, alpha=1)
plt.plot(time, sigma_vector, label="Measurement noise σ_η", color="red", linewidth=2, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.8. Comparison of final error for filtered estimate \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
P0_large = np.diag([10000.0, 10000.0])
x_true_runs_large = []
x_filt_runs_large = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0_large)
    x_true_runs_large.append(x_true)
    x_filt_runs_large.append(res["x_filt"][:, 0])

final_error_filt_large = compute_final_error(x_true_runs_large, x_filt_runs_large)

P0_small = np.diag([100.0, 100.0])
x_true_runs_small = []
x_filt_runs_small = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0_small)
    x_true_runs_small.append(x_true)
    x_filt_runs_small.append(res["x_filt"][:, 0])

final_error_filt_small = compute_final_error(x_true_runs_small, x_filt_runs_small)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_filt_large, label="Final error for filtered estimate", color="blue", linewidth=1.5, alpha=1)
plt.plot(time, final_error_filt_small, label="Filtered estimate (accurate initial filtration error)", color="orange", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.9. Comparison of FINAL ERROR for both versions of the initial filtration error covariance matrix \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


In [ ]:
x_true_runs = []
x_filt_runs = []
x_pred_runs = []
sigma_filt_runs = []
sigma_pred_runs = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0)
    
    x_true_runs.append(x_true)
    x_filt_runs.append(res["x_filt"][:, 0])
    x_pred_runs.append(res["x_pred"][:, 0])
    
    sigma_filt_runs.append(res["sigma_x_filt"])
    sigma_pred_runs.append(res["sigma_x_pred"])

final_error_filt = compute_final_error(x_true_runs, x_filt_runs)
sigma_filt_mean = np.mean(np.array(sigma_filt_runs), axis=0)[3:]

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_filt, label="True estimation error for filtered estimate", color="blue", linewidth=1.5, alpha=1)
plt.plot(time, sigma_filt_mean, label="Calculated estimation error", color="orange", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.10. Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
x_true_runs = []
x_pred_runs = []
sigma_pred_runs = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0)
    
    x_true_runs.append(x_true)
    x_pred_runs.append(res["x_pred"][:, 0])
    sigma_pred_runs.append(res["sigma_x_pred"])

final_error_pred = compute_final_error(x_true_runs, x_pred_runs)
sigma_pred_mean = np.mean(np.array(sigma_pred_runs), axis=0)[3:]

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_pred, label="True estimation error for predicted estimate", color="blue", linewidth=2, alpha=0.6)
plt.plot(time, sigma_pred_mean, label="Calculated estimation error", color="orange", linewidth=2, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.11. Prediction errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
z = generate_measurements(x_true, sigma_eta)
res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0)

K_pos = res["K"][:, 0, 0]

time = np.arange(len(K_pos))

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, K_pos, label="Kalman Gain", color="blue", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Kalman gain for estimate \n", fontsize=fontsize_label)
plt.title("\n Fig.12. Kalman Gain for deterministic trajectory \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
x_true_runs = []
x_filt_runs = []
x_pred_runs = []
sigma_filt_runs = []
sigma_pred_runs = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    res = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0)
    
    x_true_runs.append(x_true)
    x_filt_runs.append(res["x_filt"][:, 0])
    x_pred_runs.append(res["x_pred"][:, 0])
    
    sigma_filt_runs.append(res["sigma_x_filt"])
    sigma_pred_runs.append(res["sigma_x_pred"])

final_error_filt = compute_final_error(x_true_runs, x_filt_runs)
final_error_pred = compute_final_error(x_true_runs, x_pred_runs)

sigma_filt_mean = np.mean(np.array(sigma_filt_runs), axis=0)[3:]
sigma_pred_mean = np.mean(np.array(sigma_pred_runs), axis=0)[3:]

time = np.arange(len(final_error_filt))

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_filt, label="True estimation error for filtered estimate", color="blue", linewidth=1.5, alpha=0.6)
plt.plot(time, sigma_filt_mean, label="Calculated estimation error", color="orange", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.13. Filtration errors for deterministic trajectory \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


plt.figure(figsize=(25, 8))

plt.plot(time, final_error_pred, label="True estimation error for predicted estimate", color="blue", linewidth=1.5, alpha=0.6)
plt.plot(time, sigma_pred_mean, label="Calculated estimation error", color="orange", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.14. Prediction errors for deterministic trajectory \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
x_true_runs = []
x_filt_runs = []
x_future_runs = []
sigma_a_true = 0.2

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a_true)
    z = generate_measurements(x_true, sigma_eta)

    res = run_kalman(z, T, 0.0, sigma_eta, x0_hat, v0_hat, P0)
    x_future = predict_ahead_from_filtered(res["x_filt"], res["Phi"], m)

    x_true_runs.append(x_true)
    x_filt_runs.append(res["x_filt"][:, 0])
    x_future_runs.append(x_future[:, 0])

final_error_filt = compute_final_error(x_true_runs, x_filt_runs)
final_error_future = compute_final_error(x_true_runs, x_future_runs)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_filt, label="True estimation error for filtered estimate", color="blue", linewidth=1.5, alpha=0.6)
plt.plot(time, final_error_future, label="True estimation error for predicted future estimate", color="orange", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.15. Final errors for Q = 0 \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
x_true_runs = []
x_filt_runs = []
sigma_filt_runs = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a_true)
    z = generate_measurements(x_true, sigma_eta)
    
    res = run_kalman(z, T, 0.0, sigma_eta, x0_hat, v0_hat, P0)
    
    x_true_runs.append(x_true)
    x_filt_runs.append(res["x_filt"][:, 0])
    sigma_filt_runs.append(res["sigma_x_filt"])

final_error_filt = compute_final_error(x_true_runs, x_filt_runs)

sigma_filt_mean = np.mean(np.array(sigma_filt_runs), axis=0)[3:]

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_filt, label="True estimation error for filtered estimate", color="blue", linewidth=1.5, alpha=0.6)
plt.plot(time, sigma_filt_mean, label="Calculated estimation error", color="orange", linewidth=1.5, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.16. Comparison of filtration errors and standard deviation for Q = 0 \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
sigma_a_1 = 0.2

x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a_1)
z = generate_measurements(x_true, sigma_eta)
res_1 = run_kalman(z, T, sigma_a_1, sigma_eta, x0_hat, v0_hat, P0)

Kpos_02 = res_1["K"][:, 0, 0]

sigma_a_2 = 1.0

x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a_2)
z = generate_measurements(x_true, sigma_eta)
res_2 = run_kalman(z, T, sigma_a_2, sigma_eta, x0_hat, v0_hat, P0)

Kpos_1 = res_2["K"][:, 0, 0]

time = np.arange(N)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, Kpos_02, label="Kalman gain for σ_a^2 = 0.04", color="blue", linewidth=1, alpha=1)
plt.plot(time, Kpos_1,  label="Kalman gain for σ_a^2 = 1.0",  color="orange", linewidth=1, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Kalman gain for estimate \n", fontsize=fontsize_label)
plt.title("\n Fig.17. Kalman gain \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

In [ ]:
x_true_runs = []
x_filt_runs_opt = []
x_filt_runs_under = []

for _ in range(runs):
    x_true, v_true, a_true = generate_true_trajectory(N, T, x0, v0, sigma_a)
    z = generate_measurements(x_true, sigma_eta)
    
    res_opt = run_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0)
    x_filt_opt = res_opt["x_filt"][:, 0]
    
    K_under = res_opt["K"][-10] / 5.0
    Phi, G, H, Q, R = build_model_matrices(T, sigma_a, sigma_eta)
    I = np.eye(2)
    
    n = len(z)
    x_filt_under = np.zeros((n, 2))
    P_prev = P0
    x_prev = np.array([x0_hat, v0_hat])
    
    for i in range(n):
        x_pr = Phi @ x_prev
        P_pr = Phi @ P_prev @ Phi.T + Q
        resid = z[i] - (H @ x_pr).item()
        x_up = x_pr + resid * K_under.flatten()
        P_up = (I - K_under @ H) @ P_pr
        x_filt_under[i] = x_up
        x_prev, P_prev = x_up, P_up

    x_true_runs.append(x_true)
    x_filt_runs_opt.append(x_filt_opt)
    x_filt_runs_under.append(x_filt_under[:, 0])

final_error_opt = compute_final_error(x_true_runs, x_filt_runs_opt)
final_error_under = compute_final_error(x_true_runs, x_filt_runs_under)

time = np.arange(len(final_error_opt))

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(time, final_error_opt, label="True estimation error for filtered estimate", color="blue", linewidth=1, alpha=1)
plt.plot(time, final_error_under, label="True estimation error for underestimated filtered estimate", color="orange", linewidth=1, alpha=1)

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error values, m \n", fontsize=fontsize_label)
plt.title("\n Fig.18. Final errors for x0 = 100, v0 = 5 \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()